# Multi-Mode RCPSP with Time-Offs

This notebook combines the [Multi-Mode RCPSP](multimode_rcpsp.ipynb) with the [RCPSP with Time-Offs](rcpsp_timeoffs.ipynb). Each task has multiple execution modes (trading off duration vs. resource requirements), and each resource unit has an individual availability calendar. The notebook covers four scheduling variants:

1. **No Migration | No Delays**: Tasks execute continuously; each assigned unit must be available throughout.
2. **Migration | No Delays**: Tasks execute continuously but can use any available units (aggregate capacity).
3. **No Migration | Delays | Blocked**: Tasks pause when any assigned unit is unavailable; all assigned units remain blocked.
4. **Migration | Delays**: Tasks pause when aggregate capacity is insufficient; can resume on different units.

## Problem Definition

We consider a set of tasks $\mathcal{N}$ with finish-to-start precedence relations $\mathcal{E}$. Each task $i$ has a set of **execution modes** $\mathcal{M}_i$. Mode $j \in \mathcal{M}_i$ specifies:
- A processing duration $d_{ij}$
- Resource type requirements $q_{ij,k}$ (quantity of type $k$ needed)

Resources are organized into **types** $\mathcal{K}$. Each type $k$ has a set of **resource units** $\mathcal{U}_k$. Each unit $r$ has an availability calendar $\mathcal{F}_r(t) \in \{0, 1\}$.

The objective is to minimize the makespan $C_{\max}$ by choosing one execution mode per task and scheduling tasks to satisfy precedences, resource constraints, and calendar constraints.

## Parsing, Loading, Helper Functions

### Imports & Parsing

In [ ]:
import docplex.cp.utils_visu as visu
import matplotlib as plt
from itertools import combinations, product
from docplex.cp.model import CpoStepFunction

In [ ]:
def next_line(f):
    """Read next non-empty, non-comment line."""
    while True:
        raw = f.readline()
        if not raw:
            return None
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        return [int(v) for v in line.split()]

def load_instance(filename):
    """Load MMRCPSP instance with time-offs.
    Returns: (N, K, U, TASKS, TYPES, UNITS, PRECEDENCES) where
        - N: number of tasks
        - K: number of resource types
        - U: number of resource units
        - TASKS: [(task_id, [(mode_id, duration, [(type_id, qty), ...]), ...]), ...]
        - TYPES: [(type_id, [unit_ids]), ...]
        - UNITS: [(unit_id, [(time, intensity), ...]), ...]
        - PRECEDENCES: [(pred_task, succ_task), ...]
    """
    with open(filename, "r") as f:
        N, K, U = next_line(f)
        TYPES = [(d[0], d[2:2+d[1]]) for d in (next_line(f) for _ in range(K))]
        UNITS = [(d[0], [(d[2+2*i], d[3+2*i]) for i in range(d[1])])
                 for d in (next_line(f) for _ in range(U))]
        TASKS = []
        for _ in range(N):
            header = next_line(f)
            task_id, num_modes = header[0], header[1]
            modes = []
            for m in range(num_modes):
                mode_line = next_line(f)
                duration, num_reqs = mode_line[0], mode_line[1]
                reqs = [tuple(next_line(f)[:2]) for _ in range(num_reqs)]
                modes.append((m + 1, duration, reqs))
            TASKS.append((task_id, modes))
        PRECEDENCES = [tuple(next_line(f)[:2]) for _ in range(next_line(f)[0])]
    return N, K, U, TASKS, TYPES, UNITS, PRECEDENCES

### Helper Functions

In [ ]:
HORIZON = 100_000

# === Core Helpers ===
def get_availability(unit_id, time, res_map):
    """Returns availability (0 or 100) of a unit at a specific time."""
    return next((v for t, v in reversed(res_map[unit_id]) if time >= t), 0)

def step_function(steps, horizon=HORIZON):
    """Create CpoStepFunction from [(time, value), ...] pairs."""
    f = CpoStepFunction()
    for i, (t, v) in enumerate(steps):
        end = steps[i + 1][0] if i + 1 < len(steps) else horizon
        f.set_value(t, end, v)
    return f

# === Resource Processing ===
def prepare_types(TYPES):
    """type_id -> {name, units, capacity}"""
    return {tid: {"name": f"Type_{tid}", "units": units, "capacity": len(units)}
            for tid, units in TYPES}

def extract_breaks(UNITS, horizon=HORIZON):
    """Extract (start, duration) pairs where unit is unavailable."""
    return {
        uid: [(t, (steps[i+1][0] if i+1 < len(steps) else horizon) - t)
              for i, (t, v) in enumerate(steps) if v == 0
              and t < (steps[i+1][0] if i+1 < len(steps) else horizon)]
        for uid, steps in UNITS if any(v == 0 for _, v in steps)
    }

def joint_intensity(unit_ids, res_map, horizon=HORIZON):
    """CpoStepFunction: 100 only when ALL units available simultaneously."""
    if not unit_ids:
        return CpoStepFunction(steps=[(0, 100)])
    times = sorted({0} | {t for uid in unit_ids for t, _ in res_map[uid]})
    steps = [(t, 100 if all(get_availability(u, t, res_map) for u in unit_ids) else 0)
             for t in times]
    return step_function(steps, horizon)

def joint_intensity_steps(unit_ids, res_map, horizon=HORIZON):
    """Returns [(time, value), ...] where value=1 when ALL units available, else 0.
    For OptalCP step functions."""
    if not unit_ids:
        return [(0, 1)]
    times = sorted({0} | {t for uid in unit_ids if uid in res_map for t, _ in res_map[uid]})
    if not times:
        return [(0, 1)]
    return [(t, 1 if all(get_availability(u, t, res_map) > 0 for u in unit_ids) else 0)
            for t in times]

# === Mode Generation ===
def build_full_modes(TASKS, TYPE_MAP):
    """Generate all (exec_mode_id, duration, unit_combo) for each task.
    Each full mode is a combination of execution mode and specific unit assignment."""
    result = {}
    for tid, modes in TASKS:
        task_full_modes = []
        for mid, dur, reqs in modes:
            if not reqs or all(qty == 0 for _, qty in reqs):
                task_full_modes.append((mid, dur, ()))
                continue
            combos = [list(combinations(TYPE_MAP[t], q)) if q > 0 else [()]
                      for t, q in reqs]
            for c in product(*combos):
                unit_set = tuple(sorted({r for grp in c for r in grp}))
                task_full_modes.append((mid, dur, unit_set))
        result[tid] = task_full_modes
    return result

def capacity_windows(reqs, TYPE_MAP, RES_MAP, horizon=HORIZON):
    """Find windows where aggregate capacity >= requirements."""
    if not reqs or all(qty == 0 for _, qty in reqs):
        return [(0, horizon)]
    times = sorted({0, horizon} | {t for tid, qty in reqs if qty > 0
                                    for r in TYPE_MAP[tid] if r in RES_MAP
                                    for t, _ in RES_MAP[r]})
    def feasible(t):
        return all(sum(get_availability(r, t, RES_MAP) > 0 for r in TYPE_MAP[tid]) >= qty
                   for tid, qty in reqs if qty > 0)
    windows = []
    for i in range(len(times) - 1):
        if feasible(times[i]):
            if windows and windows[-1][1] == times[i]:
                windows[-1] = (windows[-1][0], times[i + 1])
            else:
                windows.append((times[i], times[i + 1]))
    return windows

### Visualisation

In [ ]:
def print_instance(N, K, U, TASKS, TYPES, UNITS, PRECEDENCES):
    print(f"\n{'='*80}")
    print(f"INSTANCE: {N} Tasks x {K} Types x {U} Units")
    print(f"{'='*80}")
    TYPE_MAP = {tid: units for tid, units in TYPES}
    print("\nResource Types:")
    for type_id, units in TYPES:
        print(f"  Type {type_id}: Units {{{', '.join(f'U{u}' for u in units)}}}")
    print(f"\n{'Task':<6} {'Modes':<6} {'Mode Details'}")
    print(f"{'-'*80}")
    for tid, modes in TASKS:
        succs = [s for p, s in PRECEDENCES if p == tid]
        mode_strs = []
        for mid, dur, reqs in modes:
            if not reqs or all(q == 0 for _, q in reqs):
                mode_strs.append(f"M{mid}(d={dur})")
            else:
                req_str = "+".join(f"{q}xT{t}" for t, q in reqs if q > 0)
                mode_strs.append(f"M{mid}(d={dur},{req_str})")
        print(f"T{tid:<5} {len(modes):<6} {', '.join(mode_strs):<50} -> {succs if succs else ''}")
    print(f"\n{'Unit':<6} {'Type':<6} {'Available Windows'}")
    print(f"{'-'*80}")
    unit_to_type = {u: tid for tid, units in TYPES for u in units}
    for unit_id, steps in UNITS:
        windows = []
        for i in range(len(steps)):
            if steps[i][1] > 0:
                end = steps[i+1][0] if i+1 < len(steps) else "inf"
                windows.append(f"[{steps[i][0]}-{end})")
        type_id = unit_to_type.get(unit_id, "?")
        print(f"U{unit_id:<5} T{type_id:<5} {' '.join(windows) if windows else 'never'}")
    print(f"{'='*80}\n")

### Load Instance

In [ ]:
from docplex.cp.model import *

In [ ]:
filename = "../data/mmrcpsp-timeoffs/mmrcpsp_timeoffs_default.data"

N, K, U, TASKS, TYPES, UNITS, PRECEDENCES = load_instance(filename)
RES_MAP = dict(UNITS)
TYPE_MAP = dict(TYPES)
print_instance(N, K, U, TASKS, TYPES, UNITS, PRECEDENCES)

## 1. No Migration | No Delays

Tasks execute continuously without interruption. Each task selects one execution mode (determining duration and resource type requirements), then is assigned to specific resource units. All assigned units must be continuously available throughout the task's execution.

### CP Formulation

$$
\begin{aligned}
\min \quad
& \max_{i \in \mathcal{N}} \operatorname{endOf}(T_i)
& & \text{(1)} \\[2mm]
\text{s.t.} \quad
& \operatorname{endBeforeStart}(T_i, T_j),
& \forall (i,j) \in \mathcal{E}
& \text{(2)} \\[2mm]
& \operatorname{alternative}(T_i, \{Y_{ij} \mid j \in \mathcal{M}_i\}),
& \forall i \in \mathcal{N}
& \text{(3)} \\[2mm]
& \operatorname{alternative}(Y_{ij}, \{O_{ijr} \mid r \in \mathcal{U}_k\},\; \text{card}=q_{ij,k}),
& \forall i,j,\; \forall k: q_{ij,k}>0
& \text{(4)} \\[2mm]
& \operatorname{noOverlap}(\{O_{ijr} \mid (i,j): r \in \mathcal{C}_{ij}\}),
& \forall r \in \mathcal{R}
& \text{(5)} \\[2mm]
& \operatorname{forbidExtent}(O_{ijr}, \mathcal{F}_r),
& \forall O_{ijr}
& \text{(6)} \\[2mm]
& T_i{:}\; \text{mandatory interval var}
& \forall i \in \mathcal{N}
& \text{(7a)} \\[1mm]
& Y_{ij}{:}\; \text{optional interval var, size}=d_{ij}
& \forall i, j \in \mathcal{M}_i
& \text{(7b)} \\[1mm]
& O_{ijr}{:}\; \text{optional interval var, size}=d_{ij}
& \forall i, j, r
& \text{(7c)}
\end{aligned}
$$

**Objective**
- **(1)** Minimize makespan.

**Constraints**
- **(2)** Precedence relations.
- **(3)** **Execution mode selection**: exactly one mode $j$ is selected for each task $i$, synchronizing $T_i$ with the chosen $Y_{ij}$.
- **(4)** **Unit selection**: for the selected mode $j$, exactly $q_{ij,k}$ units of each required type $k$ are selected. The `alternative` with cardinality synchronizes $Y_{ij}$ with the selected unit intervals.
- **(5)** **Resource exclusivity**: no two tasks use the same resource unit simultaneously.
- **(6)** **Calendar compliance**: each unit interval must fall entirely within the unit's availability windows.

**Variables**
- **(7a)** $T_i$: master interval for task $i$.
- **(7b)** $Y_{ij}$: optional interval for execution mode $j$ of task $i$.
- **(7c)** $O_{ijr}$: optional interval for task $i$, mode $j$, on unit $r$.

### IBM CPO DOcplex Implementation

In [ ]:
# Prepare data: step functions for each resource unit's calendar
res_availability = {uid: step_function(steps) for uid, steps in UNITS}

# === Create model ===
mdl = CpoModel(name="mmrcpsp_timeoffs_m1_cpo")

# (7a) T_i: mandatory master interval
T = {tid: interval_var(name=f"T{tid}") for tid, _ in TASKS}

# (7b) Y_{ij}: optional interval per execution mode
Y = {(tid, mid): interval_var(size=dur, optional=True, name=f"T{tid}_M{mid}")
     for tid, modes in TASKS for mid, dur, _ in modes}

# (7c) O_{ijr}: optional interval per mode x unit
O = {}
for tid, modes in TASKS:
    for mid, dur, reqs in modes:
        for type_id, qty in reqs:
            if qty > 0:
                for r in TYPE_MAP[type_id]:
                    O[(tid, mid, r)] = interval_var(size=dur, optional=True,
                                                    name=f"T{tid}_M{mid}_U{r}")

# === Constraints ===
# (1) Minimize makespan
mdl.add(minimize(max(end_of(T[tid]) for tid in T)))

# (2) Precedences
mdl.add([end_before_start(T[i], T[j]) for i, j in PRECEDENCES])

# (3) Execution mode selection: alternative(T_i, [Y_{ij}])
for tid, modes in TASKS:
    mdl.add(alternative(T[tid], [Y[(tid, mid)] for mid, _, _ in modes]))

# (4) Unit selection per mode and type: alternative(Y_{ij}, [...], cardinality)
for tid, modes in TASKS:
    for mid, dur, reqs in modes:
        for type_id, qty in reqs:
            if qty > 0:
                candidates = [O[(tid, mid, r)] for r in TYPE_MAP[type_id]]
                mdl.add(alternative(Y[(tid, mid)], candidates, cardinality=qty))

# (5) NoOverlap per resource unit
for r in range(U):
    if intervals := [itv for (tid, mid, unit), itv in O.items() if unit == r]:
        mdl.add(no_overlap(intervals))

# (6) Calendar compliance: forbidExtent
for (tid, mid, r), itv in O.items():
    if r in res_availability:
        mdl.add(forbid_extent(itv, res_availability[r]))

# === Solve ===
print("Solving Model 1: No Migration | No Delays (CPO)...")
if sol := mdl.solve(LogVerbosity='Quiet'):
    print(f"\n{'='*70}")
    print(f"MAKESPAN: {sol.get_objective_values()[0]}")
    print(f"{'='*70}")
    print(f"{'Task':<6} {'Mode':<6} {'Start':<7} {'End':<7} {'Dur':<5} {'Units'}")
    print(f"{'-'*70}")
    for tid, modes in TASKS:
        t_sol = sol.get_var_solution(T[tid])
        if not t_sol: continue
        start, end = t_sol.get_start(), t_sol.get_end()
        sel_mode = None
        for mid, dur, reqs in modes:
            y_sol = sol.get_var_solution(Y[(tid, mid)])
            if y_sol and y_sol.is_present():
                sel_mode = mid
                break
        units = sorted(r for (t, m, r) in O if t == tid and m == sel_mode
                       and (s := sol.get_var_solution(O[(t, m, r)])) and s.is_present())
        unit_str = "{" + ",".join(f"U{r}" for r in units) + "}" if units else "-"
        print(f"T{tid:<5} M{sel_mode if sel_mode else '?':<5} {start:<7} {end:<7} {end-start:<5} {unit_str}")
    print(f"{'='*70}")
else:
    print("No solution found.")

### OptalCP Implementation

In [ ]:
import optalcp as cp

mdl = cp.Model(name="mmrcpsp_timeoffs_m1_optal")

# Step functions must be created via the model
res_availability_op = {uid: mdl.step_function(steps) for uid, steps in UNITS}

# (7a) T_i: mandatory master interval
T = {tid: mdl.interval_var(name=f"T{tid}") for tid, _ in TASKS}

# (7b) Y_{ij}: optional interval per execution mode
Y = {(tid, mid): mdl.interval_var(length=dur, optional=True, name=f"T{tid}_M{mid}")
     for tid, modes in TASKS for mid, dur, _ in modes}

# (7c) O_{ijr}: optional interval per mode x unit
O = {}
for tid, modes in TASKS:
    for mid, dur, reqs in modes:
        for type_id, qty in reqs:
            if qty > 0:
                for r in TYPE_MAP[type_id]:
                    O[(tid, mid, r)] = mdl.interval_var(length=dur, optional=True,
                                                        name=f"T{tid}_M{mid}_U{r}")

# (1) Minimize makespan
mdl.minimize(mdl.max([T[tid].end() for tid in T]))

# (2) Precedences
mdl.enforce([T[i].end_before_start(T[j]) for i, j in PRECEDENCES])

# (3) Execution mode selection
for tid, modes in TASKS:
    mdl.enforce(mdl.alternative(T[tid], [Y[(tid, mid)] for mid, _, _ in modes]))

# (4) Unit selection: decomposed alternative with cardinality
for tid, modes in TASKS:
    for mid, dur, reqs in modes:
        for type_id, qty in reqs:
            if qty > 0:
                candidates = [O[(tid, mid, r)] for r in TYPE_MAP[type_id]]
                mdl.enforce(mdl.sum([c.presence() for c in candidates])
                            == qty * Y[(tid, mid)].presence())
                for c in candidates:
                    mdl.enforce(Y[(tid, mid)].start_at_start(c))
                    mdl.enforce(Y[(tid, mid)].end_at_end(c))

# (5) NoOverlap per resource unit
for r in range(U):
    if intervals := [itv for (tid, mid, unit), itv in O.items() if unit == r]:
        mdl.enforce(mdl.no_overlap(intervals))

# (6) Calendar compliance
for (tid, mid, r), itv in O.items():
    if r in res_availability_op:
        mdl.enforce(itv.forbid_extent(res_availability_op[r]))

# Solve
print("Solving Model 1: No Migration | No Delays (OptalCP)...")
result = mdl.solve(cp.Parameters(timeLimit=10, logLevel=0))
if result.solution:
    sol = result.solution
    print(f"\n{'='*70}\nMAKESPAN: {result.objective}\n{'='*70}")
    print(f"{'Task':<6} {'Mode':<6} {'Start':<7} {'End':<7} {'Dur':<5} {'Units'}")
    print(f"{'-'*70}")
    for tid, modes in TASKS:
        start, end = sol.get_start(T[tid]), sol.get_end(T[tid])
        sel_mode = next((mid for mid, _, _ in modes if sol.is_present(Y[(tid, mid)])), None)
        units = sorted(r for (t, m, r) in O if t == tid and m == sel_mode
                       and sol.is_present(O[(t, m, r)]))
        unit_str = "{" + ",".join(f"U{r}" for r in units) + "}" if units else "-"
        print(f"T{tid:<5} M{sel_mode if sel_mode else '?':<5} {start:<7} {end:<7} {end-start:<5} {unit_str}")
    print(f"{'='*70}")
else:
    print("No solution found.")

## 2. Migration | No Delays

Tasks execute continuously without interruption, but can use any available units of the required types (migration allowed). The model tracks aggregate capacity per resource type rather than individual unit assignments. A task is valid as long as the total available capacity of each required type meets the demand at every moment during execution.

### CP Formulation

$$
\begin{aligned}
\min \quad
& \max_{i \in \mathcal{N}} \operatorname{endOf}(T_i)
& & \text{(1)} \\[2mm]
\text{s.t.} \quad
& \operatorname{endBeforeStart}(T_i, T_j),
& \forall (i,j) \in \mathcal{E}
& \text{(2)} \\[2mm]
& \operatorname{alternative}(T_i, \{Y_{ij} \mid j \in \mathcal{M}_i\}),
& \forall i \in \mathcal{N}
& \text{(3)} \\[2mm]
& \sum_{i,j} \operatorname{pulse}(Y_{ij}, q_{ij,k}) \leq A_k,
& \forall k \in \mathcal{K}
& \text{(4)} \\[2mm]
& A_k(t) = \sum_{r \in \mathcal{U}_k} \mathcal{F}_r(t),
& \forall k \in \mathcal{K}
& \text{(5)} \\[2mm]
& T_i{:}\; \text{mandatory interval var}
& \forall i \in \mathcal{N}
& \text{(6a)} \\[1mm]
& Y_{ij}{:}\; \text{optional interval var, size}=d_{ij}
& \forall i, j \in \mathcal{M}_i
& \text{(6b)}
\end{aligned}
$$

**Constraints**
- **(3)** Execution mode selection.
- **(4)** Aggregate resource capacity: at every time $t$, the sum of resource demands from all active modes must not exceed the available capacity $A_k(t)$.
- **(5)** Available capacity is the sum of individual unit availabilities.

### IBM CPO DOcplex Implementation

In [ ]:
res_types = prepare_types(TYPES)
res_breaks = extract_breaks(UNITS)

mdl = CpoModel(name="mmrcpsp_timeoffs_m2_cpo")

# (6a) T_i: mandatory master interval
T = {tid: interval_var(name=f"T{tid}") for tid, _ in TASKS}

# (6b) Y_{ij}: optional interval per execution mode
Y = {(tid, mid): interval_var(size=dur, optional=True, name=f"T{tid}_M{mid}")
     for tid, modes in TASKS for mid, dur, _ in modes}

# (5) A_k(t): aggregate availability per type
A = {type_id: step_at(0, rtype["capacity"]) - sum(
    pulse((s, s+d), 1) for u in rtype["units"] if u in res_breaks for s, d in res_breaks[u])
    for type_id, rtype in res_types.items()}

# (1) Minimize makespan
mdl.add(minimize(max(end_of(T[tid]) for tid in T)))

# (2) Precedences
mdl.add([end_before_start(T[i], T[j]) for i, j in PRECEDENCES])

# (3) Execution mode selection
for tid, modes in TASKS:
    mdl.add(alternative(T[tid], [Y[(tid, mid)] for mid, _, _ in modes]))

# (4) Aggregate capacity constraint
for type_id in res_types:
    if pulses := [pulse(Y[(tid, mid)], qty)
                  for tid, modes in TASKS for mid, dur, reqs in modes
                  for req_type, qty in reqs if req_type == type_id and qty > 0]:
        mdl.add(A[type_id] - sum(pulses) >= 0)

# Solve
print("Solving Model 2: Migration | No Delays (CPO)...")
if sol := mdl.solve(LogVerbosity='Quiet'):
    print(f"\n{'='*70}\nMAKESPAN: {sol.get_objective_values()[0]}\n{'='*70}")
    print(f"{'Task':<6} {'Mode':<6} {'Start':<7} {'End':<7} {'Dur':<5}")
    print(f"{'-'*70}")
    for tid, modes in TASKS:
        t_sol = sol.get_var_solution(T[tid])
        if not t_sol: continue
        start, end = t_sol.get_start(), t_sol.get_end()
        sel_mode = next((mid for mid, _, _ in modes
                         if (y := sol.get_var_solution(Y[(tid, mid)])) and y.is_present()), None)
        print(f"T{tid:<5} M{sel_mode if sel_mode else '?':<5} {start:<7} {end:<7} {end-start:<5}")
    print(f"{'='*70}")
else:
    print("No solution found.")

### OptalCP Implementation

In [ ]:
import optalcp as cp

res_types = prepare_types(TYPES)
res_breaks = extract_breaks(UNITS)

mdl = cp.Model(name="mmrcpsp_timeoffs_m2_optal")

T = {tid: mdl.interval_var(name=f"T{tid}") for tid, _ in TASKS}
Y = {(tid, mid): mdl.interval_var(length=dur, optional=True, name=f"T{tid}_M{mid}")
     for tid, modes in TASKS for mid, dur, _ in modes}

# (1) Minimize makespan
mdl.minimize(mdl.max([T[tid].end() for tid in T]))

# (2) Precedences
mdl.enforce([T[i].end_before_start(T[j]) for i, j in PRECEDENCES])

# (3) Execution mode selection
for tid, modes in TASKS:
    mdl.enforce(mdl.alternative(T[tid], [Y[(tid, mid)] for mid, _, _ in modes]))

# (4) + (5) Aggregate capacity: usage + breaks <= capacity
for type_id, rtype in res_types.items():
    usage = mdl.sum([mdl.pulse(Y[(tid, mid)], qty)
                     for tid, modes in TASKS for mid, dur, reqs in modes
                     for req_type, qty in reqs if req_type == type_id and qty > 0])
    breaks = mdl.sum([mdl.pulse(mdl.interval_var(start=(s, s), end=(s+d, s+d)), 1)
                      for u in rtype["units"] if u in res_breaks
                      for s, d in res_breaks[u]])
    mdl.enforce(usage + breaks <= rtype["capacity"])

# Solve
print("Solving Model 2: Migration | No Delays (OptalCP)...")
result = mdl.solve(cp.Parameters(timeLimit=10, logLevel=0))
if result.solution:
    sol = result.solution
    print(f"\n{'='*70}\nMAKESPAN: {result.objective}\n{'='*70}")
    print(f"{'Task':<6} {'Mode':<6} {'Start':<7} {'End':<7} {'Dur':<5}")
    print(f"{'-'*70}")
    for tid, modes in TASKS:
        start, end = sol.get_start(T[tid]), sol.get_end(T[tid])
        sel_mode = next((mid for mid, _, _ in modes if sol.is_present(Y[(tid, mid)])), None)
        print(f"T{tid:<5} M{sel_mode if sel_mode else '?':<5} {start:<7} {end:<7} {end-start:<5}")
    print(f"{'='*70}")
else:
    print("No solution found.")

## 3. No Migration | Delays | Blocked

Each task selects an execution mode (determining duration and resource requirements) and is assigned to specific resource units. When any assigned unit becomes unavailable, the task pauses — but all assigned units remain blocked (unavailable to other tasks), even if some of their calendars show availability. Work accumulates only when all assigned units are simultaneously available.

### CP Formulation

We enumerate **full modes** $\mathcal{FM}_i$ for each task $i$. Each full mode $f = (j, c)$ combines an execution mode $j \in \mathcal{M}_i$ with a specific unit assignment $c$ (a set of resource units satisfying mode $j$'s requirements).

$$
\begin{aligned}
\min \quad
& \max_{i \in \mathcal{N}} \operatorname{endOf}(T_i)
& & \text{(1)} \\[2mm]
\text{s.t.} \quad
& \operatorname{endBeforeStart}(T_i, T_j),
& \forall (i,j) \in \mathcal{E}
& \text{(2)} \\[2mm]
& \operatorname{alternative}(T_i, \{O_{i,f} \mid f \in \mathcal{FM}_i\}),
& \forall i \in \mathcal{N}
& \text{(3)} \\[2mm]
& \operatorname{noOverlap}(\{O_{i,f} \mid i \in \mathcal{N}, f \in \mathcal{FM}_i{:}\; r \in \mathcal{R}_f\}),
& \forall r \in \mathcal{R}
& \text{(4)} \\[2mm]
& \operatorname{forbidStart}(O_{i,f}, \mathcal{G}_f),
& \forall i, f
& \text{(5)} \\[2mm]
& \mathcal{G}_f(t) = \min_{r \in \mathcal{R}_f} \mathcal{F}_r(t),
& \forall f
& \text{(6)} \\[2mm]
& T_i{:}\; \text{mandatory interval var}
& \forall i \in \mathcal{N}
& \text{(7a)} \\[1mm]
& O_{i,f}{:}\; \text{optional interval var, size}=d_f,\; \text{intensity}=\mathcal{G}_f
& \forall i, f \in \mathcal{FM}_i
& \text{(7b)}
\end{aligned}
$$

**Key points**
- **(3)** Selects one full mode per task — determining both execution mode (duration/requirements) AND specific unit assignment.
- **(4)** Resource exclusivity via `noOverlap`. Since the interval spans the full wall-clock time (including pauses), blocked units are covered.
- **(5)** `forbidStart` prevents starting when the joint intensity is zero.
- **(6)** Joint intensity $\mathcal{G}_f$: 100 only when ALL units in the mode are available.
- **(7b)** The `intensity` parameter on the interval var ensures that `size` equals the integral of intensity over the interval (i.e., actual work time = required duration).

### IBM CPO DOcplex Implementation

In [ ]:
full_modes = build_full_modes(TASKS, TYPE_MAP)

# Pre-compute joint intensity per unique unit combo
combo_intensity_cpo = {}
for tid in full_modes:
    for mid, dur, units in full_modes[tid]:
        if units not in combo_intensity_cpo:
            combo_intensity_cpo[units] = joint_intensity(units, RES_MAP)

mdl = CpoModel(name="mmrcpsp_timeoffs_m3_cpo")

# (7a) T_i: mandatory master interval
T = {tid: interval_var(name=f"T{tid}") for tid, _ in TASKS}

# (7b) O_{i,f}: optional interval per full mode, with intensity
O = {}
for tid in full_modes:
    for idx, (mid, dur, units) in enumerate(full_modes[tid]):
        O[(tid, mid, units)] = interval_var(
            size=dur, intensity=combo_intensity_cpo[units],
            optional=True, name=f"T{tid}_M{mid}_fm{idx}")

# (1) Minimize makespan
mdl.add(minimize(max(end_of(T[tid]) for tid in T)))

# (2) Precedences
mdl.add([end_before_start(T[i], T[j]) for i, j in PRECEDENCES])

# (3) Alternative: select one full mode per task
for tid in full_modes:
    mdl.add(alternative(T[tid], [O[(tid, mid, units)]
                                  for mid, dur, units in full_modes[tid]]))

# (4) NoOverlap per resource unit
for r in range(U):
    if intervals := [O[(tid, mid, units)]
                     for (tid, mid, units) in O if r in units]:
        mdl.add(no_overlap(intervals))

# (5) ForbidStart during unavailability
for (tid, mid, units), itv in O.items():
    if units:
        mdl.add(forbid_start(itv, combo_intensity_cpo[units]))

# Solve
print("Solving Model 3: No Migration | Delays | Blocked (CPO)...")
if sol := mdl.solve(LogVerbosity='Quiet'):
    print(f"\n{'='*70}\nMAKESPAN: {sol.get_objective_values()[0]}\n{'='*70}")
    print(f"{'Task':<6} {'Mode':<6} {'Start':<7} {'End':<7} {'Work':<6} {'Units'}")
    print(f"{'-'*70}")
    for tid, _ in TASKS:
        t_sol = sol.get_var_solution(T[tid])
        if not t_sol: continue
        start, end = t_sol.get_start(), t_sol.get_end()
        for mid, dur, units in full_modes[tid]:
            o_sol = sol.get_var_solution(O[(tid, mid, units)])
            if o_sol and o_sol.is_present():
                unit_str = "{" + ",".join(f"U{r}" for r in units) + "}" if units else "-"
                print(f"T{tid:<5} M{mid:<5} {start:<7} {end:<7} {dur:<6} {unit_str}")
                break
    print(f"{'='*70}")
else:
    print("No solution found.")

### OptalCP Implementation

In [ ]:
import optalcp as cp

full_modes = build_full_modes(TASKS, TYPE_MAP)

# Joint intensity step functions (OptalCP uses 0/1 values)
combo_steps = {}
for tid in full_modes:
    for mid, dur, units in full_modes[tid]:
        if units not in combo_steps:
            combo_steps[units] = joint_intensity_steps(units, RES_MAP)

mdl = cp.Model(name="mmrcpsp_timeoffs_m3_optal")

combo_intensity_op = {units: mdl.step_function(steps)
                      for units, steps in combo_steps.items()}

# (7a) T_i: mandatory master interval
T = {tid: mdl.interval_var(name=f"T{tid}") for tid, _ in TASKS}

# (7b) O_{i,f}: optional interval per full mode (length is free — intensity determines work)
O = {}
for tid in full_modes:
    for idx, (mid, dur, units) in enumerate(full_modes[tid]):
        O[(tid, mid, units)] = mdl.interval_var(optional=True, name=f"T{tid}_M{mid}_fm{idx}")

# (1) Minimize makespan
mdl.minimize(mdl.max([T[tid].end() for tid in T]))

# (2) Precedences
mdl.enforce([T[i].end_before_start(T[j]) for i, j in PRECEDENCES])

# (3) Alternative: select one full mode per task
for tid in full_modes:
    mdl.enforce(mdl.alternative(T[tid], [O[(tid, mid, units)]
                                         for mid, dur, units in full_modes[tid]]))

# (4) NoOverlap per resource unit
for r in range(U):
    if intervals := [O[(tid, mid, units)]
                     for (tid, mid, units) in O if r in units]:
        mdl.enforce(mdl.no_overlap(intervals))

# Work content: integral of intensity over interval == required duration
for tid in full_modes:
    for mid, dur, units in full_modes[tid]:
        if dur > 0 and units:
            work = mdl.integral(combo_intensity_op[units], O[(tid, mid, units)])
            mdl.enforce(work.guard(dur) == dur)

# (5) ForbidStart
for (tid, mid, units), itv in O.items():
    if units and units in combo_intensity_op:
        mdl.enforce(itv.forbid_start(combo_intensity_op[units]))

# Solve
print("Solving Model 3: No Migration | Delays | Blocked (OptalCP)...")
result = mdl.solve(cp.Parameters(timeLimit=10, logLevel=0))
if result.solution:
    sol = result.solution
    print(f"\n{'='*70}\nMAKESPAN: {result.objective}\n{'='*70}")
    print(f"{'Task':<6} {'Mode':<6} {'Start':<7} {'End':<7} {'Units'}")
    print(f"{'-'*70}")
    for tid, _ in TASKS:
        start, end = sol.get_start(T[tid]), sol.get_end(T[tid])
        for mid, dur, units in full_modes[tid]:
            if sol.is_present(O[(tid, mid, units)]):
                unit_str = "{" + ",".join(f"U{r}" for r in units) + "}" if units else "-"
                print(f"T{tid:<5} M{mid:<5} {start:<7} {end:<7} {unit_str}")
                break
    print(f"{'='*70}")
else:
    print("No solution found.")

## 4. Migration | Delays

Tasks can be interrupted when aggregate resource capacity drops below requirements. With migration, tasks are not locked to specific resource units. When capacity is insufficient, the task pauses; when capacity returns, it resumes (potentially on different units). Each execution mode determines the duration and per-type resource requirements.

The model pre-computes **capacity windows** for each execution mode — maximal time intervals where aggregate capacity meets that mode's requirements. Tasks are split into **segments** that must fit within these windows.

### CP Formulation

$$
\begin{aligned}
\min \quad
& \max_{i \in \mathcal{N}} \operatorname{endOf}(T_i)
& & \text{(1)} \\[2mm]
\text{s.t.} \quad
& \operatorname{endBeforeStart}(T_i, T_j),
& \forall (i,j) \in \mathcal{E}
& \text{(2)} \\[2mm]
& \operatorname{alternative}(T_i, \{Y_{ij} \mid j \in \mathcal{M}_i\}),
& \forall i \in \mathcal{N}
& \text{(3)} \\[2mm]
& \operatorname{span}(Y_{ij}, \{S_{ijw} \mid w \in \mathcal{W}_{ij}\}),
& \forall i, j
& \text{(4)} \\[2mm]
& \sum_{w} \operatorname{sizeOf}(S_{ijw}) = d_{ij} \cdot \operatorname{presenceOf}(Y_{ij}),
& \forall i, j
& \text{(5)} \\[2mm]
& S_{ijw} \subseteq [w^s, w^e),
& \forall i, j, w
& \text{(6)} \\[2mm]
& \sum_{i,j} \sum_{w} \operatorname{pulse}(S_{ijw}, q_{ij,k}) \leq A_k,
& \forall k \in \mathcal{K}
& \text{(7)} \\[2mm]
& A_k(t) = \sum_{r \in \mathcal{U}_k} \mathcal{F}_r(t)
& \forall k \in \mathcal{K}
& \text{(8)} \\[2mm]
& T_i{:}\; \text{mandatory interval var}
& & \text{(9a)} \\[1mm]
& Y_{ij}{:}\; \text{optional interval var}
& & \text{(9b)} \\[1mm]
& S_{ijw}{:}\; \text{optional interval var}
& & \text{(9c)}
\end{aligned}
$$

**Key points**
- **(3)** Selects execution mode (determining duration and resource requirements).
- **(4)** `span`: master mode interval $Y_{ij}$ spans all its segments.
- **(5)** Total work across segments equals required duration (when mode is selected).
- **(6)** Each segment must stay within its pre-computed capacity window.
- **(7)** Aggregate capacity constraint on segments prevents over-allocation.
- $\mathcal{W}_{ij}$: capacity windows for task $i$ in mode $j$ — pre-computed maximal intervals where $A_k(t) \geq q_{ij,k}$ for all required types $k$.

### IBM CPO DOcplex Implementation

In [ ]:
res_types = prepare_types(TYPES)
res_breaks = extract_breaks(UNITS)

mdl = CpoModel(name="mmrcpsp_timeoffs_m4_cpo")

# Pre-compute capacity windows per execution mode
mode_windows = {}
for tid, modes in TASKS:
    for mid, dur, reqs in modes:
        mode_windows[(tid, mid)] = capacity_windows(reqs, TYPE_MAP, RES_MAP)

# (9a) T_i: mandatory master interval
T = {tid: interval_var(name=f"T{tid}") for tid, _ in TASKS}

# (9b) Y_{ij}: optional interval per execution mode
Y = {(tid, mid): interval_var(optional=True, name=f"T{tid}_M{mid}")
     for tid, modes in TASKS for mid, dur, _ in modes}

# (9c) + (6) S_{ijw}: optional segment intervals within capacity windows
S = {}
for (tid, mid), windows in mode_windows.items():
    for w, (ws, we) in enumerate(windows):
        S[(tid, mid, w)] = interval_var(optional=True,
            start=(ws, we), end=(ws, we), name=f"T{tid}_M{mid}_seg{w}")

# (8) A_k(t): aggregate availability
A = {k: step_at(0, rt["capacity"]) - sum(pulse((s, s+d), 1)
     for u in rt["units"] if u in res_breaks for s, d in res_breaks[u])
     for k, rt in res_types.items()}

# (1) Minimize makespan
mdl.add(minimize(max(end_of(T[tid]) for tid in T)))

# (2) Precedences
mdl.add([end_before_start(T[i], T[j]) for i, j in PRECEDENCES])

# (3) Execution mode selection
for tid, modes in TASKS:
    mdl.add(alternative(T[tid], [Y[(tid, mid)] for mid, _, _ in modes]))

# (4) Span: mode interval spans its segments
for tid, modes in TASKS:
    for mid, dur, reqs in modes:
        segs = [S[(tid, mid, w)] for w in range(len(mode_windows[(tid, mid)]))]
        if segs:
            mdl.add(span(Y[(tid, mid)], segs))

# (5) Work content: sum of segment sizes = duration * presenceOf(mode)
for tid, modes in TASKS:
    for mid, dur, reqs in modes:
        mdl.add(sum(size_of(S[(tid, mid, w)], 0)
                    for w in range(len(mode_windows[(tid, mid)])))
                == dur * presence_of(Y[(tid, mid)]))

# (7) Aggregate capacity on segments
for k, rt in res_types.items():
    if pulses := [pulse(S[(tid, mid, w)], qty)
                  for tid, modes in TASKS for mid, dur, reqs in modes
                  for rk, qty in reqs if qty > 0 and rk == k
                  for w in range(len(mode_windows[(tid, mid)]))]:
        mdl.add(A[k] - sum(pulses) >= 0)

# Solve
print("Solving Model 4: Migration | Delays (CPO)...")
if sol := mdl.solve(LogVerbosity='Quiet'):
    print(f"\n{'='*70}\nMAKESPAN: {sol.get_objective_values()[0]}\n{'='*70}")
    print(f"{'Task':<6} {'Mode':<6} {'Start':<7} {'End':<7} {'Work':<6} {'Segments'}")
    print(f"{'-'*70}")
    for tid, modes in TASKS:
        t_sol = sol.get_var_solution(T[tid])
        if not t_sol: continue
        start, end = t_sol.get_start(), t_sol.get_end()
        sel_mode = next((mid for mid, _, _ in modes
                         if (y := sol.get_var_solution(Y[(tid, mid)])) and y.is_present()), None)
        if sel_mode:
            segs = [(sol.get_var_solution(S[(tid, sel_mode, w)]).get_start(),
                     sol.get_var_solution(S[(tid, sel_mode, w)]).get_end())
                    for w in range(len(mode_windows[(tid, sel_mode)]))
                    if (sv := sol.get_var_solution(S[(tid, sel_mode, w)])) and sv.is_present()]
            work = sum(e - s for s, e in segs)
            seg_str = ", ".join(f"[{s}-{e})" for s, e in segs) if segs else "-"
            print(f"T{tid:<5} M{sel_mode:<5} {start:<7} {end:<7} {work:<6} {seg_str}")
    print(f"{'='*70}")
else:
    print("No solution found.")

### OptalCP Implementation

In [ ]:
import optalcp as cp

res_types = prepare_types(TYPES)
res_breaks = extract_breaks(UNITS)

# Pre-compute capacity windows per execution mode
mode_windows = {}
for tid, modes in TASKS:
    for mid, dur, reqs in modes:
        mode_windows[(tid, mid)] = capacity_windows(reqs, TYPE_MAP, RES_MAP)

mdl = cp.Model(name="mmrcpsp_timeoffs_m4_optal")

# (9a) T_i: mandatory master interval
T = {tid: mdl.interval_var(name=f"T{tid}") for tid, _ in TASKS}

# (9b) Y_{ij}: optional interval per execution mode
Y = {(tid, mid): mdl.interval_var(optional=True, name=f"T{tid}_M{mid}")
     for tid, modes in TASKS for mid, dur, _ in modes}

# (9c) + (6) S_{ijw}: optional segment intervals
S = {}
for (tid, mid), windows in mode_windows.items():
    for w, (ws, we) in enumerate(windows):
        S[(tid, mid, w)] = mdl.interval_var(optional=True,
            start=(ws, we), end=(ws, we), name=f"T{tid}_M{mid}_seg{w}")

# (1) Minimize makespan
mdl.minimize(mdl.max([T[tid].end() for tid in T]))

# (2) Precedences
mdl.enforce([T[i].end_before_start(T[j]) for i, j in PRECEDENCES])

# (3) Execution mode selection
for tid, modes in TASKS:
    mdl.enforce(mdl.alternative(T[tid], [Y[(tid, mid)] for mid, _, _ in modes]))

# (4) Span: mode interval spans its segments
for tid, modes in TASKS:
    for mid, dur, reqs in modes:
        segs = [S[(tid, mid, w)] for w in range(len(mode_windows[(tid, mid)]))]
        if segs:
            mdl.enforce(mdl.span(Y[(tid, mid)], segs))

# (5) Work content: sum of segment lengths = duration when mode is present
for tid, modes in TASKS:
    for mid, dur, reqs in modes:
        mdl.enforce(mdl.sum([S[(tid, mid, w)].length().guard(0)
                             for w in range(len(mode_windows[(tid, mid)]))])
                    == dur * Y[(tid, mid)].presence())

# (7) Aggregate capacity on segments
for type_id, rtype in res_types.items():
    usage = mdl.sum([mdl.pulse(S[(tid, mid, w)], qty)
                     for tid, modes in TASKS for mid, dur, reqs in modes
                     for rk, qty in reqs if qty > 0 and rk == type_id
                     for w in range(len(mode_windows[(tid, mid)]))])
    breaks = mdl.sum([mdl.pulse(mdl.interval_var(start=(s, s), end=(s+d, s+d)), 1)
                      for u in rtype["units"] if u in res_breaks
                      for s, d in res_breaks[u]])
    mdl.enforce(usage + breaks <= rtype["capacity"])

# Solve
print("Solving Model 4: Migration | Delays (OptalCP)...")
result = mdl.solve(cp.Parameters(timeLimit=10, logLevel=0))
if result.solution:
    sol = result.solution
    print(f"\n{'='*70}\nMAKESPAN: {result.objective}\n{'='*70}")
    print(f"{'Task':<6} {'Mode':<6} {'Start':<7} {'End':<7} {'Segments'}")
    print(f"{'-'*70}")
    for tid, modes in TASKS:
        start, end = sol.get_start(T[tid]), sol.get_end(T[tid])
        sel_mode = next((mid for mid, _, _ in modes if sol.is_present(Y[(tid, mid)])), None)
        if sel_mode:
            segs = [(sol.get_start(S[(tid, sel_mode, w)]),
                     sol.get_end(S[(tid, sel_mode, w)]))
                    for w in range(len(mode_windows[(tid, sel_mode)]))
                    if sol.is_present(S[(tid, sel_mode, w)])]
            seg_str = ", ".join(f"[{s}-{e})" for s, e in segs) if segs else "-"
            print(f"T{tid:<5} M{sel_mode:<5} {start:<7} {end:<7} {seg_str}")
    print(f"{'='*70}")
else:
    print("No solution found.")